In [15]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
from IPython.display import Markdown
from anthropic import Anthropic
from src.utils.chat import (
    add_user_message,
    add_assistant_message,
    chat_extended,
)

load_dotenv()


True

In [16]:
client = Anthropic()
model = "claude-sonnet-4-6"

In [ ]:
import json


def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. 
    
    The dataset will be used to evaluate prompts that generate Python, JSON, or 
    Regex specifically for AWS-related tasks. Generate an array of JSON objects, 
    each representing a task that requires Python, JSON, or Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code
    * Do no tinclude explanations before or after the JSON array.
    * keep the required code short
    * Make every task AWS-related

    Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)

    text = chat_extended(client, model=model, messages=messages, temperature=0)

    try:

        dataset = json.loads(text)

    except json.JSONDecodeError as exc:

        raise ValueError(

            f"Claude did not return valid JSON.\n\nRaw response:\n{text}"

        ) from exc

    if len(dataset) != 3:

        raise ValueError(

            f"Expected exactly 3 dataset entries, received {len(dataset)}."

        )

    for index, item in enumerate(dataset):

        if not isinstance(item, dict) or not isinstance(item.get("task"), str):

            raise ValueError(

                f"Dataset item {index} must be an object containing a string 'task'."

            )

    return dataset


In [14]:
dataset = generate_dataset()
dataset

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'This model does not support assistant message prefill. The conversation must end with a user message.'}, 'request_id': 'req_011CdeAa3DjwmDoNZbHeT2Ae'}